# Milestone 8: LGD Sensitivity and Realized Loss Proxy Comparison

This notebook tests how sensitive preliminary Expected Loss is to LGD definition choices and compares baseline Expected Loss against available realized loss proxies. The goal is output generation and quality checks, not final business interpretation.


## 1. Setup and Inputs

Load the prior milestone outputs and validate the required fields. Large files are loaded with selected columns where possible.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except ImportError:
    sns = None
    plt.style.use("seaborn-v0_8-whitegrid")

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")


In [ ]:
current_dir = Path.cwd()
if (current_dir / "data" / "processed").exists():
    project_root = current_dir
elif (current_dir.parent / "data" / "processed").exists():
    project_root = current_dir.parent
else:
    raise FileNotFoundError("Could not locate the project data/processed directory.")

processed_dir = project_root / "data" / "processed"
figures_dir = project_root / "reports" / "figures"
processed_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

paths = {
    "loans_clean": processed_dir / "loans_clean_v1.csv",
    "defaulted_lgd": processed_dir / "defaulted_loans_lgd_v1.csv",
    "combined": processed_dir / "loans_with_pd_lgd_ead_v1.csv",
}

for name, path in paths.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing required input: {path}")
    print(f"{name}: {path}")


In [ ]:
def available_columns(path):
    return pd.read_csv(path, nrows=0).columns.tolist()

clean_columns = available_columns(paths["loans_clean"])
defaulted_columns = available_columns(paths["defaulted_lgd"])
combined_columns = available_columns(paths["combined"])

useful_optional_fields = [
    "id", "default_flag", "funded_amnt", "ead_proxy", "recoveries", "recoveries_filled",
    "missing_recoveries_flag", "lgd", "collection_recovery_fee", "total_pymnt",
    "total_rec_prncp", "total_rec_int", "total_rec_late_fee", "loan_amnt", "grade",
    "sub_grade", "term", "purpose", "issue_d", "issue_year", "loan_status",
    "predicted_pd", "lgd_estimate", "expected_loss_prelim",
]

clean_usecols = [col for col in useful_optional_fields if col in clean_columns]
defaulted_usecols = [col for col in useful_optional_fields if col in defaulted_columns]
combined_usecols = [col for col in useful_optional_fields if col in combined_columns]

loans_clean = pd.read_csv(paths["loans_clean"], usecols=clean_usecols, low_memory=False)
defaulted_lgd = pd.read_csv(paths["defaulted_lgd"], usecols=defaulted_usecols, low_memory=False)
combined = pd.read_csv(paths["combined"], usecols=combined_usecols, low_memory=False)

print(f"loans_clean shape: {loans_clean.shape}")
print(f"defaulted_lgd shape: {defaulted_lgd.shape}")
print(f"combined PD/LGD/EAD shape: {combined.shape}")


In [ ]:
required_defaulted_any = {
    "default_flag": ["default_flag"],
    "ead": ["ead_proxy", "funded_amnt"],
    "recoveries": ["recoveries_filled", "recoveries"],
    "lgd": ["lgd"],
}
required_combined = ["predicted_pd", "ead_proxy", "lgd_estimate", "expected_loss_prelim", "default_flag"]

missing_defaulted = []
for label, options in required_defaulted_any.items():
    if not any(col in defaulted_lgd.columns for col in options):
        missing_defaulted.append(f"{label}: one of {options}")
if missing_defaulted:
    raise ValueError(f"defaulted_loans_lgd_v1.csv missing essential fields: {missing_defaulted}")

missing_combined = [col for col in required_combined if col not in combined.columns]
if missing_combined:
    raise ValueError(f"loans_with_pd_lgd_ead_v1.csv missing required fields: {missing_combined}")

validation = pd.DataFrame([
    {"check": "combined predicted_pd between 0 and 1", "passed": combined["predicted_pd"].between(0, 1).all()},
    {"check": "combined lgd_estimate between 0 and 1", "passed": combined["lgd_estimate"].between(0, 1).all()},
    {"check": "combined ead_proxy non-negative", "passed": (combined["ead_proxy"] >= 0).all()},
    {"check": "combined expected_loss_prelim non-negative", "passed": (combined["expected_loss_prelim"] >= 0).all()},
    {"check": "combined required fields have no missing values", "passed": combined[required_combined].notna().all().all()},
])
display(validation)


## 2. Prepare Defaulted Loan Sensitivity Base

Defaulted loans are used to estimate alternative LGD definitions. Rows with invalid EAD denominators are flagged and excluded from ratio calculations.


In [ ]:
defaulted_base = defaulted_lgd.copy()

# Enrich from combined data when defaulted_lgd does not include optional recovery/payment fields.
if "id" in defaulted_base.columns and "id" in combined.columns:
    missing_enrich_cols = [
        col for col in ["collection_recovery_fee", "total_pymnt", "total_rec_prncp",
                        "total_rec_int", "total_rec_late_fee", "loan_amnt", "issue_year"]
        if col in combined.columns and col not in defaulted_base.columns
    ]
    enrich_cols = ["id"] + missing_enrich_cols
    if missing_enrich_cols:
        enrich = combined.loc[combined["default_flag"] == 1, enrich_cols].drop_duplicates("id")
        defaulted_base = defaulted_base.merge(enrich, on="id", how="left")

if "ead_proxy" not in defaulted_base.columns:
    defaulted_base["ead_proxy"] = defaulted_base["funded_amnt"]

if "recoveries_filled" not in defaulted_base.columns:
    defaulted_base["missing_recoveries_flag"] = defaulted_base["recoveries"].isna()
    defaulted_base["recoveries_filled"] = defaulted_base["recoveries"].fillna(0)
elif "missing_recoveries_flag" not in defaulted_base.columns:
    defaulted_base["missing_recoveries_flag"] = defaulted_base["recoveries_filled"].isna()

numeric_cols = [
    "ead_proxy", "funded_amnt", "recoveries", "recoveries_filled", "collection_recovery_fee",
    "total_pymnt", "total_rec_prncp", "total_rec_int", "total_rec_late_fee", "loan_amnt",
]
for col in numeric_cols:
    if col in defaulted_base.columns:
        defaulted_base[col] = pd.to_numeric(defaulted_base[col], errors="coerce")

if "issue_year" not in defaulted_base.columns and "issue_d" in defaulted_base.columns:
    defaulted_base["issue_year"] = pd.to_datetime(defaulted_base["issue_d"], errors="coerce").dt.year

if "default_flag" in defaulted_base.columns:
    defaulted_base = defaulted_base.loc[defaulted_base["default_flag"] == 1].copy()

defaulted_base["invalid_ead_flag"] = defaulted_base["ead_proxy"].isna() | (defaulted_base["ead_proxy"] <= 0)
print(f"Defaulted sensitivity base shape: {defaulted_base.shape}")
print(f"Invalid EAD rows: {defaulted_base['invalid_ead_flag'].sum():,}")
print(f"Missing recoveries before fill: {defaulted_base['missing_recoveries_flag'].sum():,}")



## 3. Alternative LGD Definitions

Each LGD definition keeps raw and clipped values. Clipping constrains LGD to the practical range from 0 to 1 for Expected Loss scenario use.


In [ ]:
lgd_definitions = []

def safe_ratio(numerator, denominator):
    ratio = numerator / denominator
    return ratio.where(~defaulted_base["invalid_ead_flag"])

def add_lgd_definition(name, raw_lgd, rate_col=None):
    raw_col = f"{name}_raw"
    clipped_col = name
    flag_col = f"{name}_clipped_flag"
    defaulted_base[raw_col] = raw_lgd
    defaulted_base[clipped_col] = raw_lgd.clip(0, 1)
    defaulted_base[flag_col] = raw_lgd.lt(0) | raw_lgd.gt(1)
    defaulted_base.loc[defaulted_base["invalid_ead_flag"], [raw_col, clipped_col]] = np.nan
    defaulted_base.loc[defaulted_base["invalid_ead_flag"], flag_col] = False
    lgd_definitions.append({"name": name, "raw_col": raw_col, "clipped_col": clipped_col, "flag_col": flag_col, "rate_col": rate_col})
    print(f"Created {name}")

# A. Baseline gross recovery LGD
recovery_rate_baseline = safe_ratio(defaulted_base["recoveries_filled"], defaulted_base["ead_proxy"])
defaulted_base["baseline_recovery_rate"] = recovery_rate_baseline
add_lgd_definition("lgd_baseline", 1 - recovery_rate_baseline, "baseline_recovery_rate")

# B. Net recovery LGD after collection recovery fee
if "collection_recovery_fee" in defaulted_base.columns:
    defaulted_base["collection_recovery_fee"] = defaulted_base["collection_recovery_fee"].fillna(0)
    defaulted_base["net_recoveries"] = defaulted_base["recoveries_filled"] - defaulted_base["collection_recovery_fee"]
    defaulted_base["net_recovery_rate_fee"] = safe_ratio(defaulted_base["net_recoveries"], defaulted_base["ead_proxy"])
    add_lgd_definition("lgd_net_recovery_fee", 1 - defaulted_base["net_recovery_rate_fee"], "net_recovery_rate_fee")
else:
    print("Skipping lgd_net_recovery_fee because collection_recovery_fee is unavailable.")

# C. Principal recovery LGD
if "total_rec_prncp" in defaulted_base.columns:
    defaulted_base["principal_recovery_rate"] = safe_ratio(defaulted_base["total_rec_prncp"], defaulted_base["ead_proxy"])
    add_lgd_definition("lgd_principal_recovery", 1 - defaulted_base["principal_recovery_rate"], "principal_recovery_rate")
else:
    print("Skipping lgd_principal_recovery because total_rec_prncp is unavailable.")

# D. Total payment LGD
if "total_pymnt" in defaulted_base.columns:
    defaulted_base["total_payment_rate"] = safe_ratio(defaulted_base["total_pymnt"], defaulted_base["ead_proxy"])
    add_lgd_definition("lgd_total_payment", 1 - defaulted_base["total_payment_rate"], "total_payment_rate")
else:
    print("Skipping lgd_total_payment because total_pymnt is unavailable.")

# E. Exploratory net cashflow LGD
if all(col in defaulted_base.columns for col in ["total_pymnt", "recoveries_filled", "collection_recovery_fee"]):
    defaulted_base["total_cash_recovery_proxy"] = (
        defaulted_base["total_pymnt"].fillna(0)
        + defaulted_base["recoveries_filled"].fillna(0)
        - defaulted_base["collection_recovery_fee"].fillna(0)
    )
    defaulted_base["net_cashflow_rate"] = safe_ratio(defaulted_base["total_cash_recovery_proxy"], defaulted_base["ead_proxy"])
    add_lgd_definition("lgd_net_cashflow", 1 - defaulted_base["net_cashflow_rate"], "net_cashflow_rate")
else:
    print("Skipping lgd_net_cashflow because total_pymnt, recoveries, or collection_recovery_fee is unavailable.")

print("LGD definitions created:", [d["name"] for d in lgd_definitions])


In [ ]:
saved_outputs = {}

def save_output(df, filename):
    path = processed_dir / filename
    df.to_csv(path, index=False)
    saved_outputs[filename] = {"path": path, "shape": df.shape}
    print(f"Saved {filename}: {df.shape}")

save_output(defaulted_base, "lgd_sensitivity_defaulted_loans_v1.csv")


## 4. LGD Sensitivity Summary

Summarize each available LGD definition across defaulted loans.


In [ ]:
summary_rows = []
for definition in lgd_definitions:
    col = definition["clipped_col"]
    flag_col = definition["flag_col"]
    rate_col = definition["rate_col"]
    valid = defaulted_base[col].dropna()
    summary_rows.append({
        "lgd_definition": col,
        "valid_count": valid.shape[0],
        "mean_lgd": valid.mean(),
        "median_lgd": valid.median(),
        "std_lgd": valid.std(),
        "p05_lgd": valid.quantile(0.05),
        "p25_lgd": valid.quantile(0.25),
        "p75_lgd": valid.quantile(0.75),
        "p95_lgd": valid.quantile(0.95),
        "min_lgd": valid.min(),
        "max_lgd": valid.max(),
        "share_lgd_eq_1": (valid == 1).mean(),
        "share_lgd_eq_0": (valid == 0).mean(),
        "share_clipped": defaulted_base.loc[valid.index, flag_col].mean(),
        "mean_recovery_or_payment_rate": defaulted_base.loc[valid.index, rate_col].mean() if rate_col else np.nan,
    })

lgd_sensitivity_summary = pd.DataFrame(summary_rows)
save_output(lgd_sensitivity_summary, "lgd_sensitivity_summary_v1.csv")
display(lgd_sensitivity_summary)


## 5. Portfolio Expected Loss Sensitivity

Hold PD and EAD fixed, then apply each defaulted-loan portfolio-average LGD to all loans.


In [ ]:
total_ead = combined["ead_proxy"].sum()
baseline_total_el = combined["expected_loss_prelim"].sum()

scenario_rows = [{
    "scenario_name": "milestone_6_baseline_expected_loss_prelim",
    "portfolio_avg_lgd": combined["lgd_estimate"].mean(),
    "total_expected_loss": baseline_total_el,
    "mean_expected_loss": combined["expected_loss_prelim"].mean(),
    "median_expected_loss": combined["expected_loss_prelim"].median(),
    "expected_loss_rate": baseline_total_el / total_ead,
    "difference_vs_baseline_total_el": 0.0,
    "percent_difference_vs_baseline_total_el": 0.0,
}]

for definition in lgd_definitions:
    name = definition["clipped_col"]
    portfolio_avg_lgd = defaulted_base[name].mean()
    scenario_el = combined["predicted_pd"] * portfolio_avg_lgd * combined["ead_proxy"]
    total_scenario_el = scenario_el.sum()
    scenario_rows.append({
        "scenario_name": name,
        "portfolio_avg_lgd": portfolio_avg_lgd,
        "total_expected_loss": total_scenario_el,
        "mean_expected_loss": scenario_el.mean(),
        "median_expected_loss": scenario_el.median(),
        "expected_loss_rate": total_scenario_el / total_ead,
        "difference_vs_baseline_total_el": total_scenario_el - baseline_total_el,
        "percent_difference_vs_baseline_total_el": (total_scenario_el - baseline_total_el) / baseline_total_el,
    })

expected_loss_lgd_sensitivity_summary = pd.DataFrame(scenario_rows)
save_output(expected_loss_lgd_sensitivity_summary, "expected_loss_lgd_sensitivity_summary_v1.csv")
display(expected_loss_lgd_sensitivity_summary)


## 6. Realized Loss Proxy Construction

Create realized loss proxies for defaulted loans using available cash recovery and payment fields. These are proxies, not accounting losses.


In [ ]:
realized = combined.copy()
for col in ["ead_proxy", "recoveries", "recoveries_filled", "collection_recovery_fee", "total_rec_prncp", "total_pymnt"]:
    if col in realized.columns:
        realized[col] = pd.to_numeric(realized[col], errors="coerce")

if "recoveries_filled" not in realized.columns:
    if "recoveries" not in realized.columns:
        raise ValueError("Cannot create realized gross recovery loss without recoveries or recoveries_filled.")
    realized["recoveries_filled"] = realized["recoveries"].fillna(0)
else:
    realized["recoveries_filled"] = realized["recoveries_filled"].fillna(0)

realized_loss_proxy_cols = []

def add_realized_proxy(col_name, recovery_amount):
    raw_col = f"{col_name}_raw"
    realized[raw_col] = realized["default_flag"] * (realized["ead_proxy"] - recovery_amount)
    realized[col_name] = realized[raw_col].clip(lower=0)
    realized_loss_proxy_cols.append(col_name)
    print(f"Created {col_name}")

add_realized_proxy("realized_loss_gross_recovery", realized["recoveries_filled"])

if "collection_recovery_fee" in realized.columns:
    realized["collection_recovery_fee"] = realized["collection_recovery_fee"].fillna(0)
    add_realized_proxy("realized_loss_net_recovery_fee", realized["recoveries_filled"] - realized["collection_recovery_fee"])
else:
    print("Skipping realized_loss_net_recovery_fee because collection_recovery_fee is unavailable.")

if "total_rec_prncp" in realized.columns:
    add_realized_proxy("realized_loss_principal", realized["total_rec_prncp"].fillna(0))
else:
    print("Skipping realized_loss_principal because total_rec_prncp is unavailable.")

if "total_pymnt" in realized.columns:
    add_realized_proxy("realized_loss_total_payment", realized["total_pymnt"].fillna(0))
else:
    print("Skipping realized_loss_total_payment because total_pymnt is unavailable.")

print("Realized loss proxy columns:", realized_loss_proxy_cols)


## 7. Realized vs Expected Comparison

Compare baseline preliminary Expected Loss to each available realized loss proxy at portfolio level.


In [ ]:
comparison_rows = []
for proxy_col in realized_loss_proxy_cols:
    total_realized = realized[proxy_col].sum()
    comparison_rows.append({
        "realized_loss_proxy": proxy_col,
        "total_expected_loss_prelim": baseline_total_el,
        "total_realized_loss_proxy": total_realized,
        "difference_expected_minus_realized": baseline_total_el - total_realized,
        "ratio_expected_to_realized": baseline_total_el / total_realized if total_realized != 0 else np.nan,
        "expected_loss_rate": baseline_total_el / total_ead,
        "realized_loss_rate_proxy": total_realized / total_ead,
    })

realized_vs_expected_loss_summary = pd.DataFrame(comparison_rows)
save_output(realized_vs_expected_loss_summary, "realized_vs_expected_loss_summary_v1.csv")
display(realized_vs_expected_loss_summary)


## 8. Segment Realized Loss Proxy Summaries

Use gross recovery realized loss as the main proxy for grade and PD-decile comparisons.


In [ ]:
main_realized_proxy = "realized_loss_gross_recovery"

def ratio_expected_to_realized(expected, realized_amount):
    return expected / realized_amount if realized_amount != 0 else np.nan

def realized_segment_summary(df, group_col):
    grouped = (
        df.groupby(group_col, dropna=False)
        .agg(
            loan_count=("expected_loss_prelim", "size"),
            default_count=("default_flag", "sum"),
            mean_predicted_pd=("predicted_pd", "mean"),
            realized_default_rate=("default_flag", "mean"),
            total_ead=("ead_proxy", "sum"),
            total_expected_loss_prelim=("expected_loss_prelim", "sum"),
            total_realized_loss_proxy=(main_realized_proxy, "sum"),
        )
        .reset_index()
    )
    grouped["expected_loss_rate"] = grouped["total_expected_loss_prelim"] / grouped["total_ead"]
    grouped["realized_loss_proxy_rate"] = grouped["total_realized_loss_proxy"] / grouped["total_ead"]
    grouped["difference_expected_minus_realized"] = grouped["total_expected_loss_prelim"] - grouped["total_realized_loss_proxy"]
    grouped["ratio_expected_to_realized"] = grouped["total_expected_loss_prelim"] / grouped["total_realized_loss_proxy"].replace({0: np.nan})
    return grouped

if "grade" in realized.columns:
    realized_loss_proxy_by_grade = realized_segment_summary(realized, "grade")
    realized_loss_proxy_by_grade["grade"] = pd.Categorical(realized_loss_proxy_by_grade["grade"], categories=list("ABCDEFG"), ordered=True)
    realized_loss_proxy_by_grade = realized_loss_proxy_by_grade.sort_values("grade")
    save_output(realized_loss_proxy_by_grade, "realized_loss_proxy_by_grade_v1.csv")
    display(realized_loss_proxy_by_grade)
else:
    print("Skipping realized loss by grade because grade is unavailable.")

realized["pd_decile"] = pd.qcut(realized["predicted_pd"], q=10, labels=False, duplicates="drop") + 1
realized_loss_proxy_by_pd_decile = realized_segment_summary(realized, "pd_decile")
realized_loss_proxy_by_pd_decile = realized_loss_proxy_by_pd_decile.sort_values("pd_decile")
save_output(realized_loss_proxy_by_pd_decile, "realized_loss_proxy_by_pd_decile_v1.csv")
display(realized_loss_proxy_by_pd_decile)


## 9. Vintage LGD Sensitivity

Summarize alternative LGD definitions by issue year when vintage data is available.


In [ ]:
if "issue_year" not in defaulted_base.columns and "issue_d" in defaulted_base.columns:
    defaulted_base["issue_year"] = pd.to_datetime(defaulted_base["issue_d"], errors="coerce").dt.year

if "issue_year" in defaulted_base.columns:
    vintage_group = defaulted_base.dropna(subset=["issue_year"]).copy()
    vintage_group["issue_year"] = vintage_group["issue_year"].astype(int)
    vintage_rows = []
    for issue_year, group in vintage_group.groupby("issue_year"):
        row = {"issue_year": issue_year, "default_count": len(group)}
        for definition in lgd_definitions:
            name = definition["clipped_col"]
            row[f"mean_{name}"] = group[name].mean()
            row[f"median_{name}"] = group[name].median()
            rate_col = definition["rate_col"]
            if rate_col:
                row[f"mean_{rate_col}"] = group[rate_col].mean()
        vintage_rows.append(row)
    lgd_sensitivity_by_vintage = pd.DataFrame(vintage_rows).sort_values("issue_year")
    save_output(lgd_sensitivity_by_vintage, "lgd_sensitivity_by_vintage_v1.csv")
    display(lgd_sensitivity_by_vintage)
else:
    print("Skipping vintage LGD sensitivity because issue_year or issue_d is unavailable.")


## 10. Plots

Create concise visual checks for LGD definitions, Expected Loss scenarios, realized loss comparisons, and vintage sensitivity.


In [ ]:
def save_current_fig(filename):
    path = figures_dir / filename
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved figure: {path}")

plt.figure(figsize=(9, 5))
plot_df = lgd_sensitivity_summary.sort_values("mean_lgd", ascending=False)
plt.bar(plot_df["lgd_definition"], plot_df["mean_lgd"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("Mean LGD")
plt.title("Mean LGD by Definition")
save_current_fig("mean_lgd_by_definition_v1.png")

lgd_cols = [d["clipped_col"] for d in lgd_definitions]
lgd_plot_sample = defaulted_base[lgd_cols].sample(min(50000, len(defaulted_base)), random_state=42).melt(var_name="lgd_definition", value_name="lgd")
plt.figure(figsize=(10, 5))
if sns is not None:
    sns.boxplot(data=lgd_plot_sample, x="lgd_definition", y="lgd")
else:
    lgd_plot_sample.boxplot(column="lgd", by="lgd_definition", rot=30)
plt.xticks(rotation=30, ha="right")
plt.ylabel("LGD")
plt.title("LGD Distribution by Definition")
save_current_fig("lgd_distribution_by_definition_v1.png")

plt.figure(figsize=(10, 5))
plot_df = expected_loss_lgd_sensitivity_summary.sort_values("total_expected_loss", ascending=False)
plt.bar(plot_df["scenario_name"], plot_df["total_expected_loss"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("Total Expected Loss")
plt.title("Portfolio Total Expected Loss by LGD Scenario")
save_current_fig("total_expected_loss_by_lgd_scenario_v1.png")

if "realized_loss_proxy_by_grade" in globals():
    grade_plot = realized_loss_proxy_by_grade.copy()
    x = np.arange(len(grade_plot))
    width = 0.4
    plt.figure(figsize=(9, 5))
    plt.bar(x - width / 2, grade_plot["total_expected_loss_prelim"], width, label="Expected Loss")
    plt.bar(x + width / 2, grade_plot["total_realized_loss_proxy"], width, label="Realized Loss Proxy")
    plt.xticks(x, grade_plot["grade"])
    plt.ylabel("Loss Amount")
    plt.title("Expected Loss vs Realized Gross Recovery Loss by Grade")
    plt.legend()
    save_current_fig("expected_vs_realized_loss_by_grade_v1.png")

pd_plot = realized_loss_proxy_by_pd_decile.copy()
x = np.arange(len(pd_plot))
width = 0.4
plt.figure(figsize=(10, 5))
plt.bar(x - width / 2, pd_plot["total_expected_loss_prelim"], width, label="Expected Loss")
plt.bar(x + width / 2, pd_plot["total_realized_loss_proxy"], width, label="Realized Loss Proxy")
plt.xticks(x, pd_plot["pd_decile"])
plt.xlabel("PD Decile")
plt.ylabel("Loss Amount")
plt.title("Expected Loss vs Realized Gross Recovery Loss by PD Decile")
plt.legend()
save_current_fig("expected_vs_realized_loss_by_pd_decile_v1.png")

if "lgd_sensitivity_by_vintage" in globals():
    plt.figure(figsize=(11, 5))
    for definition in lgd_definitions:
        name = definition["clipped_col"]
        plt.plot(lgd_sensitivity_by_vintage["issue_year"], lgd_sensitivity_by_vintage[f"mean_{name}"], marker="o", label=name)
    plt.xlabel("Issue Year")
    plt.ylabel("Mean LGD")
    plt.title("Mean LGD by Vintage and Definition")
    plt.legend()
    save_current_fig("mean_lgd_by_vintage_definition_v1.png")


## 11. Output Checks

Print saved output paths, shapes, and key Milestone 8 sensitivity results.


In [ ]:
output_check = pd.DataFrame([
    {"file": name, "path": str(info["path"]), "shape": info["shape"], "exists": info["path"].exists()}
    for name, info in saved_outputs.items()
])
display(output_check)

baseline_lgd = expected_loss_lgd_sensitivity_summary.loc[
    expected_loss_lgd_sensitivity_summary["scenario_name"] == "milestone_6_baseline_expected_loss_prelim",
    "portfolio_avg_lgd",
].iloc[0]

print(f"Baseline portfolio-average LGD: {baseline_lgd:.4f}")
print("Alternative portfolio-average LGDs:")
for _, row in expected_loss_lgd_sensitivity_summary.iterrows():
    print(f"  {row['scenario_name']}: LGD={row['portfolio_avg_lgd']:.4f}, Total EL={row['total_expected_loss']:,.2f}")

if "realized_loss_gross_recovery" in realized_loss_proxy_cols:
    gross_realized = realized["realized_loss_gross_recovery"].sum()
    ratio = baseline_total_el / gross_realized if gross_realized != 0 else np.nan
    print(f"Total realized gross recovery loss proxy: {gross_realized:,.2f}")
    print(f"Ratio of baseline expected loss to realized gross recovery loss proxy: {ratio:.4f}")
